In [ ]:
# Install required packages (if not already installed)
%pip install databricks-sdk --upgrade mlflow --quiet
dbutils.library.restartPython()

## Import the Deployment Module

In [ ]:
# Add the src directory to the path to import our custom modules
import sys

sys.path.append("/Workspace/path/to/ADB/NOTEBOOKS/src")

from deployment.deploy_endpoint import deploy_model_endpoint, get_endpoint_status

## Configuration

Set your model and endpoint parameters here.

In [ ]:
# Configuration parameters
CATALOG = "analytics_uc"
SCHEMA = "ml_models"
MODEL_NAME = "customer_churn_predictor"
ENDPOINT_NAME = "churn-prediction-api"

# Get token from Databricks secrets (recommended)
# Replace 'my-scope' with your actual secret scope name
TOKEN = dbutils.secrets.get(scope="my-scope", key="databricks-token")

# Optional: Additional configuration
WORKLOAD_SIZE = "Small"  # Options: Small, Medium, Large
SCALE_TO_ZERO = True

# Tags for organizing endpoints
TAGS = {
    "team": "data-science",
    "project": "customer-retention",
    "environment": "production",
    "model_type": "classification",
}

## Deploy Model to Serving Endpoint

This will create a new endpoint or update an existing one with the latest model version.

In [ ]:
try:
    print("🚀 Starting model deployment...")

    result = deploy_model_endpoint(
        token=TOKEN,
        catalog=CATALOG,
        schema=SCHEMA,
        model_name=MODEL_NAME,
        endpoint_name=ENDPOINT_NAME,
        workload_size=WORKLOAD_SIZE,
        scale_to_zero_enabled=SCALE_TO_ZERO,
        tags=TAGS,
    )

    print("\n✅ Deployment completed successfully!")
    print(f"📍 Endpoint URL: {result['endpoint_url']}")
    print(f"📦 Model Version: {result['model_version']}")
    print(f"🏷️  Full Model Name: {result['full_model_name']}")

except Exception as e:
    print(f"❌ Deployment failed: {e}")
    raise

## Verify Endpoint Status

Check the status of the deployed endpoint.

In [ ]:
# Check endpoint status
try:
    status = get_endpoint_status(ENDPOINT_NAME, TOKEN)

    print("📊 Endpoint Status:")
    print(f"   Name: {status['name']}")
    print(f"   State: {status['state']}")
    print(f"   Creator: {status['creator']}")
    print(f"   Created: {status['creation_timestamp']}")
    print(f"\n   Tags:")
    for key, value in status["tags"].items():
        print(f"      {key}: {value}")

except Exception as e:
    print(f"⚠️  Could not retrieve status: {e}")

## Test the Endpoint

Send a test request to verify the endpoint is working correctly.

In [ ]:
import requests
import json


# Example: Test the endpoint with sample data
def test_endpoint(endpoint_name, token, sample_data):
    """
    Send a test request to the serving endpoint.

    Args:
        endpoint_name: Name of the endpoint
        token: Databricks token
        sample_data: Dictionary with 'dataframe_records' or 'dataframe_split' format
    """
    workspace_url = spark.conf.get("spark.databricks.workspaceUrl")
    url = f"https://{workspace_url}/serving-endpoints/{endpoint_name}/invocations"

    headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

    response = requests.post(url, headers=headers, json=sample_data)

    if response.status_code == 200:
        print("✅ Endpoint test successful!")
        return response.json()
    else:
        print(f"❌ Endpoint test failed: {response.status_code}")
        print(f"Response: {response.text}")
        return None


# Example test data - adjust based on your model's input schema
sample_data = {
    "dataframe_records": [{"feature1": 1.0, "feature2": 2.5, "feature3": "category_a"}]
}

# Run the test
prediction = test_endpoint(ENDPOINT_NAME, TOKEN, sample_data)
if prediction:
    print(f"\n📈 Prediction result: {prediction}")

## Integration with Training Pipeline

Here's how to integrate deployment at the end of your training notebook:

In [ ]:
# Example: Complete training + deployment flow
"""
# At the end of your training notebook:

# 1. Train your model
model = train_model(X_train, y_train)

# 2. Register model to Unity Catalog
import mlflow
mlflow.set_registry_uri("databricks-uc")

with mlflow.start_run():
    mlflow.sklearn.log_model(
        model,
        "model",
        registered_model_name=f"{CATALOG}.{SCHEMA}.{MODEL_NAME}"
    )

# 3. Deploy to serving endpoint
from deployment.deploy_endpoint import deploy_model_endpoint

result = deploy_model_endpoint(
    token=dbutils.secrets.get("my-scope", "databricks-token"),
    catalog=CATALOG,
    schema=SCHEMA,
    model_name=MODEL_NAME,
    endpoint_name=ENDPOINT_NAME,
    tags={"training_date": str(datetime.now())}
)

print(f"🎉 Model trained and deployed! Access at: {result['endpoint_url']}")
"""

## Advanced: Deploy Specific Model Version

In [ ]:
# If you want to deploy a specific version instead of the latest:

SPECIFIC_VERSION = "3"  # Replace with your desired version

result = deploy_model_endpoint(
    token=TOKEN,
    catalog=CATALOG,
    schema=SCHEMA,
    model_name=MODEL_NAME,
    endpoint_name=f"{ENDPOINT_NAME}-v{SPECIFIC_VERSION}",
    model_version=SPECIFIC_VERSION,
    workload_size="Medium",
    scale_to_zero_enabled=False,
    tags={"version": SPECIFIC_VERSION, "deployment_type": "rollback"},
)

print(f"✅ Deployed version {SPECIFIC_VERSION} to: {result['endpoint_url']}")

## Clean Up (Optional)

Delete an endpoint if needed (use with caution!).

In [ ]:
# Uncomment to delete an endpoint
# from deployment.deploy_endpoint import delete_endpoint
#
# delete_endpoint(
#     endpoint_name="old-endpoint-name",
#     token=TOKEN
# )